## Importing Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
import os

## Defining constants and file paths

In [2]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

base_dir = './dataset'
train_dir = os.path.join(base_dir, 'train/train')
val_dir = os.path.join(base_dir, 'Sample_fake_images/Sample_fake_images')
test_dir = os.path.join(base_dir, 'test/test')

## Data Augmentation and Preprocessing

In [3]:
train_datagen = ImageDataGenerator(
    rotation_range=20,           # Slight rotations
    width_shift_range=0.1,       # Slight horizontal shifts
    height_shift_range=0.1,      # Slight vertical shifts
    shear_range=0.1,             # Shearing transformations
    zoom_range=0.1,              # Slight zooming in/out
    horizontal_flip=True,        # Flip images horizontally
    fill_mode='nearest',          # Strategy for filling in newly created pixels
    validation_split=0.20         # Reserve 20% of the training data for validation
)

test_datagen = ImageDataGenerator() # No augmentation for test data, just rescaling if needed


## Loading Data using flow_from_directory

In [4]:
print("Loading Training Data:")
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH), # Resizes all images to 224x224
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    subset='training'                    # Use the 80% training subset
)

print("Loading Validation Data:")
validation_generator = train_datagen.flow_from_directory(
    train_dir,                           # Point to the Train folder again!
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False,
    subset='validation'                  # Uses the reserved 20% half
)

print("Loading Test Data:")
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False                        # No need to shuffle test data
)

Loading Training Data:
Found 384 images belonging to 2 classes.
Loading Validation Data:
Found 95 images belonging to 2 classes.
Loading Test Data:
Found 499 images belonging to 2 classes.


In [5]:
base_model = EfficientNetB0(
    weights='imagenet',       # Use weights trained on millions of images
    include_top=False,        # Drop the original 1000-class ImageNet classifier
    input_shape=(224, 224, 3)
)

# 2. Freeze the base model layers (Crucial for the first phase of training)
base_model.trainable = False

# 3. Build our custom classification head on top of it
x = base_model.output
x = GlobalAveragePooling2D()(x)       # Flattens the output efficiently
x = BatchNormalization()(x)
x = Dropout(0.5)(x)                   # Heavy dropout to prevent overfitting
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x) # Binary output (Real/Fake)

# 4. Combine into the final model
model = Model(inputs=base_model.input, outputs=predictions)

# 5. Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,713,124 (17.98 MB)

 Trainable params: 659,969 (2.52 MB)

 Non-trainable params: 4,053,155 (15.46 MB)

## Training the model with callbacks

In [6]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,          # Stop if validation loss doesn't improve for 5 epochs
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,          # Reduce learning rate by 80% if it plateaus
    patience=3,
    min_lr=1e-6
)

In [7]:
from sklearn.utils import class_weight
import numpy as np

train_classes = train_generator.classes

class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_classes),
    y=train_classes
)

class_weight_dict = dict(enumerate(class_weights_array))

print(f"Calculated Class Weights: {class_weight_dict}")

Calculated Class Weights: {0: np.float64(1.5609756097560976), 1: np.float64(0.735632183908046)}


In [8]:
history = model.fit(
    train_generator,
    epochs=20, # Early stopping will likely halt this earlier
    validation_data=validation_generator,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weight_dict
)

Epoch 1/20
 9/12 ━━━━━━━━━━━━━━━━━━━━ 1s 649ms/step - accuracy: 0.4629 - loss: 1.1438

/Users/skakibahammed/code_playground/Model_Repo/image_detect/.venv/lib/python3.13/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


12/12 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.5521 - loss: 1.0134 - val_accuracy: 0.5789 - val_loss: 0.6784 - learning_rate: 0.0010
Epoch 2/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 11s 886ms/step - accuracy: 0.7214 - loss: 0.6093 - val_accuracy: 0.6421 - val_loss: 0.6658 - learning_rate: 0.0010
Epoch 3/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 771ms/step - accuracy: 0.7526 - loss: 0.5502 - val_accuracy: 0.6842 - val_loss: 0.6173 - learning_rate: 0.0010
Epoch 4/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 783ms/step - accuracy: 0.8073 - loss: 0.4410 - val_accuracy: 0.7474 - val_loss: 0.5652 - learning_rate: 0.0010
Epoch 5/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 762ms/step - accuracy: 0.8099 - loss: 0.4540 - val_accuracy: 0.7579 - val_loss: 0.5388 - learning_rate: 0.0010
Epoch 6/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 774ms/step - accuracy: 0.8333 - loss: 0.4001 - val_accuracy: 0.7474 - val_loss: 0.4686 - learning_rate: 0.0010
Epoch 7/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 801ms/step - accuracy: 0.8438 - loss: 0.3247 - val_accura

## Evaluating the model on the test dataset

In [9]:
def evaluate_on_test_set(model, test_gen):
    print("Evaluating model on the test dataset...")
    test_loss, test_accuracy = model.evaluate(test_gen)

    print("-" * 30)
    print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
    print(f"Test Loss:     {test_loss:.4f}")
    print("-" * 30)

In [10]:
evaluate_on_test_set(model, test_generator)

Evaluating model on the test dataset...
16/16 ━━━━━━━━━━━━━━━━━━━━ 9s 542ms/step - accuracy: 0.7475 - loss: 0.5451
------------------------------
Test Accuracy: 74.75%
Test Loss:     0.5451
------------------------------


## Fine-tuning the model by unfreezing some layers of the base model

In [11]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # 1e-5 is 100x smaller than your previous 1e-3
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

print("Starting Fine-Tuning Phase...")
history_finetune = model.fit(
    train_generator,
    epochs=15,
    validation_data=validation_generator,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weight_dict
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,713,124 (17.98 MB)

 Trainable params: 2,156,129 (8.22 MB)

 Non-trainable params: 2,556,995 (9.75 MB)

Starting Fine-Tuning Phase...
Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.8490 - loss: 0.3732 - val_accuracy: 0.7895 - val_loss: 0.4885 - learning_rate: 1.0000e-05
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.8568 - loss: 0.3274 - val_accuracy: 0.8000 - val_loss: 0.4678 - learning_rate: 1.0000e-05
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.8542 - loss: 0.3822 - val_accuracy: 0.7579 - val_loss: 0.4918 - learning_rate: 1.0000e-05
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.8568 - loss: 0.3574 - val_accuracy: 0.7684 - val_loss: 0.5291 - learning_rate: 2.0000e-06
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.8516 - loss: 0.4143 - val_accuracy: 0.7684 - val_loss: 0.5055 - learning_rate: 2.0000e-06


In [12]:
evaluate_on_test_set(model, test_generator)

Evaluating model on the test dataset...
16/16 ━━━━━━━━━━━━━━━━━━━━ 10s 633ms/step - accuracy: 0.7455 - loss: 0.5481
------------------------------
Test Accuracy: 74.55%
Test Loss:     0.5481
------------------------------
